In [1]:
from core.data_constructor import DataConstructor
from datetime import datetime

/Users/stefanvandeweerdt/Projects/wec-first-draft/ML2Grow - inference/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Loads the dataset from snowflake - retrieves lat / long via geocoding (nominatim) - performs further geo mappings.
data_constructor = DataConstructor(TCBB_enable=False, address_enabled=True)
df = data_constructor.create_feature_set()

finised initialisation
created DossierReader
created VerweerderReader
created TCBBReader
created Datacollector
[dossiers] Logic hash changed (6ebfa4dbf81f4856 -> 99949a1ab90c3f41).
Checking if new logic hash 99949a1ab90c3f41 already exists
Rerunning query and saving
[SnapshotManager] Saved data to: data/dossiers_99949a1ab90c3f41.parquet
[SnapshotManager] Saved metadata to: data/dossiers_99949a1ab90c3f41.metadata.txt
[dossiers] New snapshot saved with version: 99949a1ab90c3f41
[dossiers] apply_initial_filter
Applying transfrom : get_cumulative_bedrag
Applying transfrom : add_categorical_features
Applying transfrom : add_label_features
unique hoofddossier_ids:  1
get_features DossierReader DONE
[verweerders] Logic hash changed (3e03637b6c5d2aae -> fd5836fc04af5898).
Checking if new logic hash fd5836fc04af5898 already exists
Rerunning query and saving
[SnapshotManager] Saved data to: data/verweerders_fd5836fc04af5898.parquet
[SnapshotManager] Saved metadata to: data/verweerders_fd5836fc04

100%|██████████| 187/187 [00:01<00:00, 144.34it/s]

merge address features into verweerder features DONE
merge features verweerders DONE


This should generate following columns:
```
Index(['hoofddossier_id', 'hoofddossier_flag', 'factuurbedrag',
       'betaling_klant', 'bedrag', 'booking_code', 'booking_dut',
       'dossier_status', 'dossier_status_code', 'solvabiliteit_dossier',
       'dossier_solvabiliteit', 'creatiedatum_hoofddossier', 'datumboeking',
       'creatiedatum_dossierlijn', 'gerechterlijk_minnelijk',
       'bedrag_weighted', 'bedrag_cumulatief', 'gerechtelijk', 'y',
       'dossier_status_label_reg', 'dossier_status_label_cat', 'verweerder',
       'verweerder_id', 'verweerder_b2b_b2c', 'verweerder_leeftijd',
       'verweerder_leeftijdscategorie', 'gevangenis_key',
       'verweerder_postcode', 'verweerder_straat', 'verweerder_straat_clean',
       'verweerder_nr', 'verweerder_adres', 'verweerder_gemeente',
       'verweerder_provincie', 'verweerder_gewest',
       'verweerder_straat_clean_nunique', 'verweerder_postcode_nunique',
       'gevangenis_key_nunique', 'n_people_at_address', 'estimated_leeftijd',
       'business', 'adres', 'latitude', 'longitude', 'X_Lambert72',
       'Y_Lambert72', 'geometry', 'CD_YEAR', 'CD_MUNTY_REFNIS', 'CD_SECTOR',
       'MS_NBR_NON_ZERO_INC', 'MS_TOT_NET_TAXABLE_INC',
       'MS_AVG_TOT_NET_TAXABLE_INC', 'MS_MEDIAN_NET_TAXABLE_INC',
       'MS_INT_QUART_DIFF', 'MS_INT_QUART_COEFF', 'MS_INT_QUART_ASSYM',
       'TX_SECTOR_DESCR_NL', 'TX_SECTOR_DESCR_FR'],
      dtype='object')
```

In [3]:
from sklearn.ensemble import RandomForestClassifier
import joblib
import numpy as np

# NOTE: you will probably need to edit dossiers_query.sql because it now only queries for dossiers that have concluded
# For inference, you want the dossiers that are still open (i would suggest new + had a change)
random_forest_classifier = joblib.load('random_forest_classifier.pkl')

# The model was trained on a specific, already-cleaned/encoded feature subset (see feature_names_in_),
# not the full raw feature set from create_feature_set() - select exactly those columns, in order.
# feature_names_in_ has latitude/longitude twice (an artifact of the training pipeline), and sklearn
# now rejects a DataFrame with duplicate column names, so convert to a plain numpy array instead.
feature_names = list(random_forest_classifier.feature_names_in_)
test = df[feature_names].to_numpy()

y_hat = random_forest_classifier.predict_proba(test)
# pure prediction
# 1 = solvabel
# 0 = insolvabel

# 0.0 - 0.25 = level 0 => 0.62 kans insolvabel
# 0.25 - 0.5 = level 1 => 0.50 kans op insolvabel
# 0.5 - 0.75 = level 2 => 0.41 kans op insolvabel
# 0.75 - 1.0 = level 3 => 0.21 kans op insolvabel


/Users/stefanvandeweerdt/Projects/wec-first-draft/ML2Grow - inference/.venv/lib/python3.13/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.8.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/stefanvandeweerdt/Projects/wec-first-draft/ML2Grow - inference/.venv/lib/python3.13/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.8.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/stefanvandeweerdt/Projects/wec-first-draft/ML2Grow - inferenc

In [4]:
# Single solvency probability + stoplight level per case, with identifying columns for review/export.
p_solvabel = y_hat[:, 1]  # classifier.classes_ == [0, 1] -> column 1 = P(solvabel)

stoplight = np.zeros(len(p_solvabel), dtype=int)
stoplight[p_solvabel > 0.25] += 1
stoplight[p_solvabel > 0.5] += 1
stoplight[p_solvabel > 0.75] += 1

results = df[['hoofddossier_id', 'verweerder_id', 'verweerder']].copy()
results['p_solvabel'] = p_solvabel
results['stoplight'] = stoplight

print(results['stoplight'].value_counts().sort_index())
results[results['p_solvabel'] < 1].sort_values('p_solvabel', ascending=False).head(100)


stoplight
2     3
3    29
Name: count, dtype: int64


,hoofddossier_id,verweerder_id,verweerder,p_solvabel,stoplight
16,1450000000000121067,1450000000000078290,COTTEGNIE Dirk Roger Frieda Maria,0.90,3
21,1450000000000121067,1450000000000078290,COTTEGNIE Dirk Roger Frieda Maria,0.90,3
4,1450000000000121067,1450000000000078290,COTTEGNIE Dirk Roger Frieda Maria,0.90,3
5,1450000000000121067,1450000000000078290,COTTEGNIE Dirk Roger Frieda Maria,0.90,3
20,1450000000000121067,1450000000000078290,COTTEGNIE Dirk Roger Frieda Maria,0.90,3
25,1450000000000121067,1450000000000078290,COTTEGNIE Dirk Roger Frieda Maria,0.88,3
12,1450000000000121067,1450000000000078290,COTTEGNIE Dirk Roger Frieda Maria,0.86,3
19,1450000000000121067,1450000000000078290,COTTEGNIE Dirk Roger Frieda Maria,0.86,3
14,1450000000000121067,1450000000000078290,COTTEGNIE Dirk Roger Frieda Maria,0.86,3
13,1450000000000121067,1450000000000078290,COTTEGNIE Dirk Roger Frieda Maria,0.86,3


In [ ]:
# Write results back to Snowflake.
# TABLE_NAME / WRITE_MODE are the two things to review before running this cell.
TABLE_NAME = "INFERENCE_RESULTS"
SCHEMA_NAME = "INFERENCE"
DATABASE_NAME = "DEV"
WRITE_MODE = "replace"  # "append" = keep a full history of every run (adds run_timestamp)
                       # "replace" = truncate + load, table only ever holds the latest run

from datetime import timezone

import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

from config.snowflake_config import SnowflakeConfig

snowflake_config = SnowflakeConfig()

results_to_write = results.copy()
results_to_write["run_timestamp"] = datetime.now(timezone.utc)
# write_pandas maps DataFrame columns to Snowflake columns by name (upper-cased by convention)
results_to_write.columns = [c.upper() for c in results_to_write.columns]

conn_kwargs = dict(
    user=snowflake_config.user,
    account=snowflake_config.account,
    database=snowflake_config.database,
    schema=snowflake_config.schema,
    warehouse=snowflake_config.warehouse,
    role=snowflake_config.role,
)
if snowflake_config.use_key_pair_auth:
    conn_kwargs.update(snowflake_config.connect_args())  # {"private_key": ...}
else:
    conn_kwargs["password"] = snowflake_config.password

# write_pandas uses Snowflake's bulk PUT + COPY INTO under the hood, much faster than
# df.to_sql()'s row-by-row INSERTs for a dataframe this size.
with snowflake.connector.connect(**conn_kwargs) as conn:
    success, num_chunks, num_rows, _ = write_pandas(
        conn,
        results_to_write,
        database=DATABASE_NAME,
        schema=SCHEMA_NAME,
        table_name=TABLE_NAME,
        auto_create_table=True,
        overwrite=(WRITE_MODE == "replace"),
    )

print(f"Wrote {num_rows} rows to {TABLE_NAME} (mode='{WRITE_MODE}')")
